# CLV-Residual LightGCN — Colab 실행기

H&M 2년 또는 Dunnhumby에서 미래 90일 구매가치로 지도학습한 고객 임베딩을 LightGCN 사용자 표현에 residual로 결합합니다. 기본값은 **seed 42, validation only**이며 test 정답은 만들지 않습니다.

실행 순서: GPU 런타임 선택 → Drive 데이터 확인 → 설정 전체 검토 → 마지막 실행 승인 플래그를 `True`로 변경합니다.

## 1. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/clv-m2-lightgcn-runner'
GIT_REF = 'feat/clv-residual-lightgcn'  # main 병합 후에는 'main'으로 변경 가능
!if test -d {REPO}/.git; then git -C {REPO} fetch origin {GIT_REF} && git -C {REPO} switch {GIT_REF} && git -C {REPO} pull --ff-only; else git clone --branch {GIT_REF} https://github.com/jung-un/clv-m2-lightgcn-runner.git {REPO}; fi
%pip install -q pandas numpy scipy scikit-learn
%cd {REPO}
import os, sys
module_path = os.path.join(REPO, 'lightgcn_clv_residual.py')
assert os.path.isfile(module_path), f'새 모듈이 없습니다: {module_path}. 이 설정 셀을 다시 실행하세요.'
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('✓ 저장소와 Python import 경로 확인:', module_path)

## 2. 실행 설정

`DATASET`만 `hm` 또는 `dunnhumby`로 바꿉니다. 아래 screening 설정에서는 test·holdout을 계산하지 않습니다.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import json, os, sys, torch
REPO = globals().get('REPO', '/content/clv-m2-lightgcn-runner')
assert Path(REPO, 'lightgcn_clv_residual.py').is_file(), '1번 환경 설정 셀을 먼저 다시 실행하세요.'
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from lightgcn_clv_residual import configure_residual_run, preflight_summary, run_experiment
import lightgcn_clv_v3 as v3

DATASET = 'dunnhumby'  # 'hm' 또는 'dunnhumby'
cfg = configure_residual_run(
    DATASET,
    seed_list=(42,),
    eval_test=False,
    eval_holdout=False,
    include_constant_control=True,
    out_dir=f'/content/drive/MyDrive/논문/data/results_clv_residual_{DATASET}',
    m1_checkpoint_dir=f'/content/drive/MyDrive/논문/data/results_v3_{DATASET}',
)
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

## 3. 데이터·설정 사전검사 (학습 없음)

In [ ]:
schema = v3.SCHEMA[DATASET]
required_paths = [Path(schema['tx_path']), Path(schema['item_meta_path'])]
missing = [str(p) for p in required_paths if not p.exists()]
assert not missing, 'Drive에 필요한 원본 파일이 없습니다: ' + ', '.join(missing)
assert cfg.seed_list == (42,), 'screening은 seed 42로만 시작합니다.'
assert not cfg.eval_test and not cfg.eval_holdout, 'screening에서 test/holdout을 켜지 마세요.'
assert cfg.lambda_eval == (0.0, 0.05, 0.1, 0.25, 0.5, 1.0)
print('✓ 데이터 경로, seed, split 보호, lambda 후보 확인')
print('출력 폴더:', cfg.out_dir)

## 4. 고비용 실행 승인

위 출력 전체를 한 번에 검토한 뒤에만 아래 값을 `True`로 바꾸세요. 이 셀 전까지 모델 학습은 시작되지 않습니다.

In [ ]:
ACKNOWLEDGE_HIGH_COST = False
assert ACKNOWLEDGE_HIGH_COST, '설정 검토 후 ACKNOWLEDGE_HIGH_COST=True로 바꾸세요.'
result_df = run_experiment(cfg)

## 5. validation 결과 확인

In [ ]:
display(result_df.sort_values(['model_id', 'split', 'lambda']))
print('결과 파일:')
for path in sorted(Path(cfg.out_dir).glob('clv_residual_*')):
    print(' -', path)

## 다음 단계

두 데이터셋의 seed-42 validation에서 accuracy guardrail을 통과하고 경제적 가중 적중값이 개선된 경우에만 별도 확증 실행에서 `seed_list=(42,43,44)`, `eval_test=True`를 사용합니다. validation 결과를 본 뒤 lambda 후보나 성공조건을 바꾸지 않습니다.